# SolarDrive — End-to-End Transformer Tracking Benchmark (Section VI)

Computes the MeMOTR and MOTRv2 results for Table V of the paper.

| Cell | Output | Paper reference |
|---|---|---|
| 1 | GPU check + gradient checkpointing flag | — |
| 2 | Mount Drive, unzip dataset | — |
| 3 | **Clone & compile MeMOTR** (Deformable DETR CUDA ops) | — |
| 4 | Download MeMOTR DanceTrack checkpoint | — |
| 5 | **Clone & compile MOTRv2** (same CUDA ops, same patches) | — |
| 6 | Download MOTRv2 DanceTrack checkpoint | — |
| 7 | Clone & patch TrackEval | — |
| 8 | **Configuration** (edit only here) | — |
| 9 | Detect label format & build MOTChallenge ground truth | Section IV.B |
| 10 | **Run MeMOTR inference** (end-to-end, no YOLO proposals) | **Table V** |
| 11 | Convert MeMOTR output & apply evaluation filters | **Table V** |
| 12 | Run TrackEval (MeMOTR) | **Table V** |
| 13 | Pre-download YOLO model (for MOTRv2 proposals) | — |
| 14 | **Generate YOLO proposals & build DanceTrack structure** | **Table V** |
| 15 | **Run MOTRv2 inference** | **Table V** |
| 16 | Convert MOTRv2 output & apply evaluation filters | **Table V** |
| 17 | Coordinate sanity check | — |
| 18 | Run TrackEval (MOTRv2) | **Table V** |
| 19 | Save all results to Drive | — |

**Architecture comparison:**
- **MeMOTR** (ICCV 2023): Fully end-to-end — raw images in, track IDs out. Adds a long-term memory module that propagates track embeddings via exponential smoothing: `M_t = λ·O_t + (1-λ)·M_{t-1}`.
- **MOTRv2** (CVPR 2023): End-to-end transformer that *accepts YOLO detections as proposals*, bootstrapping the attention mechanism with strong spatial priors.

Both use DanceTrack checkpoints. No fine-tuning on SolarDrive was performed.

**Evaluation standards (identical to TbD notebook — fair cross-architecture comparison):**
- Image resolution: 2064 × 1544
- `conf ≥ 0.50` — MOT17 / MOT20 / DanceTrack standard
- `min_height ≥ 50 px` — DanceTrack (CVPR 2022) standard
- Box dimensions clamped to image bounds

**Drive layout expected:**
```
MyDrive/
  SolarDrive_dataset.zip   ← images  (SolarDrive_dataset/images/<seq>/left_camera/)
  SolarDrive_labels.zip             ← labels  (SolarDrive_dataset/labels/<seq>/left_camera/)
```

**Before running:** `Runtime → Change runtime type → T4 GPU`  
Run cells top to bottom.
- Cells 3 & 5 (CUDA compilation): ~3 min each
- Cell 10 (MeMOTR inference): ~30–45 min on T4 (gradient checkpointing overhead)
- Cell 14 (YOLO proposals): ~10–15 min on T4
- Cell 15 (MOTRv2 inference): ~15–25 min on T4

---

## Cell 1 — Verify GPU

Also sets the `USE_CHECKPOINT` flag used by MeMOTR. Both transformers require ≥32 GB VRAM for standard inference; gradient checkpointing makes them fit in T4's 15.6 GB by recomputing activations on the backward pass instead of storing them.

In [1]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', vram, 'GB')
    if vram < 32:
        print(f'  ⚠  VRAM {vram} GB < 32 GB — will use --use-checkpoint (gradient checkpointing)')
        USE_CHECKPOINT = True
    else:
        USE_CHECKPOINT = False
else:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type → T4 GPU')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB
  ⚠  VRAM 15.6 GB < 32 GB — will use --use-checkpoint (gradient checkpointing)


## Cell 2 — Mount Drive & Unzip Dataset

In [2]:
from google.colab import drive
import os, shutil, glob, subprocess

drive.mount('/content/drive')

# ── Edit these two paths to match your Google Drive ────────────────────────────
IMAGES_ZIP = '/content/drive/MyDrive/SolarDrive_dataset.zip'
LABELS_ZIP = '/content/drive/MyDrive/SolarDrive_labels.zip'
# ───────────────────────────────────────────────────────────────────────────────

DATASET = '/content/SolarDrive_dataset'
SEQS    = ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']
os.makedirs(DATASET, exist_ok=True)

if not os.path.exists(f'{DATASET}/images'):
    print('Copying images zip ...')
    shutil.copy(IMAGES_ZIP, '/content/SolarDrive_dataset.zip')
    os.system('unzip -q /content/SolarDrive_dataset.zip -d /content/')
    result = subprocess.run(
        ['unzip', '-Z', '-1', '/content/SolarDrive_dataset.zip'],
        capture_output=True, text=True
    )
    top_folder = result.stdout.strip().split('\n')[0].split('/')[0]
    extracted = f'/content/{top_folder}'
    if top_folder and extracted != DATASET and os.path.exists(extracted):
        print(f'Renaming extracted folder: {top_folder} → SolarDrive_dataset')
        os.rename(extracted, DATASET)
    print('Done.')
else:
    print('Images already unzipped — skipping.')

if not os.path.exists(f'{DATASET}/labels'):
    print('Copying labels zip ...')
    shutil.copy(LABELS_ZIP, '/content/SolarDrive_labels.zip')
    os.system(f'unzip -q /content/SolarDrive_labels.zip -d {DATASET}/')
    print('Done.')
else:
    print('Labels already unzipped — skipping.')

def find_dir(base, seq, kind):
    for path in [
        f'{base}/{kind}/{seq}/left_camera',
        f'{base}/{kind}/{seq}',
    ]:
        if os.path.isdir(path): return path
    return None

IMG_DIRS, LBL_DIRS = {}, {}
all_ok = True
print()
for seq in SEQS:
    img_dir = find_dir(DATASET, seq, 'images')
    lbl_dir = find_dir(DATASET, seq, 'labels')
    IMG_DIRS[seq] = img_dir
    LBL_DIRS[seq] = lbl_dir
    n_imgs = len(glob.glob(f'{img_dir}/*.*')) if img_dir else 0
    n_lbls = len(glob.glob(f'{lbl_dir}/*.txt')) if lbl_dir else 0
    ok = '✅' if img_dir and lbl_dir else '❌'
    print(f'  {ok} {seq}: {n_imgs} images  |  {n_lbls} label files')
    if not img_dir or not lbl_dir: all_ok = False

print()
if all_ok:
    print('✅ All sequences found.')
else:
    raise RuntimeError('Some paths missing — check paths above.')

Mounted at /content/drive
Copying images zip ...
Renaming extracted folder: tartuglare_colab → SolarDrive_dataset
Done.
Copying labels zip ...
Done.

  ✅ sun_glare_0: 836 images  |  836 label files
  ✅ sun_glare_1: 247 images  |  247 label files
  ✅ sun_glare_2: 323 images  |  323 label files
  ✅ sun_glare_3: 1046 images  |  1046 label files

✅ All sequences found.


## Cell 3 — Clone MeMOTR, Install Dependencies, Patch CUDA & Compile

MeMOTR uses Deformable DETR multi-scale attention CUDA ops. These ops were written against the
old PyTorch API (`value.type()`) and must be patched for PyTorch 2.x before compilation.
Also fixes the `LD_LIBRARY_PATH` required to load `libc10.so` at runtime.

**Runtime:** ~3 min on T4.

In [3]:
import os, subprocess, sys, glob, ctypes

MEMOTR = '/content/MeMOTR'

# ── 1. Clone ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(MEMOTR):
    print('Cloning MeMOTR ...')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/MCG-NJU/MeMOTR.git', MEMOTR], check=True)
else:
    print('MeMOTR already cloned — skipping.')

# ── 2. Install Python dependencies ────────────────────────────────────────────────────────────
print('Installing dependencies ...')
subprocess.run([
    'pip', 'install', '-q',
    'scipy', 'tqdm', 'tensorboard',
    'opencv-python', 'matplotlib', 'pyyaml',
    'pycocotools', 'gdown'
], check=True)
print('  ✅ Dependencies installed')

# ── 3. Patch PyTorch 2.x CUDA API ─────────────────────────────────────────────────────────────
print('Patching CUDA source ...')
cuda_files = glob.glob(f'{MEMOTR}/**/ms_deform_attn_cuda.cu', recursive=True)
if not cuda_files:
    raise FileNotFoundError(f'CUDA file not found in {MEMOTR}')
cuda_file = cuda_files[0]

with open(cuda_file) as f:
    content = f.read()
content = content.replace('value.type()', 'value.scalar_type()')
content = content.replace('value.scalar_type().is_cuda()', 'value.is_cuda()')
with open(cuda_file, 'w') as f:
    f.write(content)

assert 'value.type()' not in content,            'Patch 1 failed'
assert 'scalar_type().is_cuda()' not in content, 'Patch 2 failed'
print('  ✅ CUDA patches applied')

# ── 4. Compile Deformable Attention extension ───────────────────────────────────────────
print('Compiling CUDA ops (~3 min) ...')
ops_dir = os.path.dirname(cuda_file)
setup_dir = ops_dir
while setup_dir != '/' and not os.path.exists(f'{setup_dir}/setup.py'):
    setup_dir = os.path.dirname(setup_dir)
if not os.path.exists(f'{setup_dir}/setup.py'):
    make_sh = glob.glob(f'{MEMOTR}/models/ops/make.sh')
    if make_sh:
        result_c = subprocess.run(
            ['sh', 'make.sh'],
            cwd=os.path.dirname(make_sh[0]),
            capture_output=True, text=True
        )
    else:
        raise FileNotFoundError('Neither setup.py nor make.sh found')
else:
    result_c = subprocess.run(
        ['python3', 'setup.py', 'build', 'install'],
        cwd=setup_dir, capture_output=True, text=True
    )
if result_c.returncode != 0:
    print(result_c.stdout[-2000:]); print(result_c.stderr[-2000:])
    raise RuntimeError('CUDA compilation failed')
print('  ✅ CUDA ops compiled')

# ── 5. Fix LD_LIBRARY_PATH (libc10.so) ──────────────────────────────────────────────────────────
import torch
torch_lib = os.path.join(os.path.dirname(torch.__file__), 'lib')
if torch_lib not in os.environ.get('LD_LIBRARY_PATH', ''):
    os.environ['LD_LIBRARY_PATH'] = torch_lib + ':' + os.environ.get('LD_LIBRARY_PATH', '')
for lib in ['libc10.so', 'libtorch_cpu.so', 'libtorch.so']:
    lp = os.path.join(torch_lib, lib)
    if os.path.exists(lp):
        try: ctypes.CDLL(lp)
        except OSError: pass

egg_paths = (
    glob.glob('/usr/local/lib/python*/dist-packages/MultiScaleDeformableAttention*.egg') +
    glob.glob(f'{setup_dir}/dist/MultiScaleDeformableAttention*.egg')
)
for ep in egg_paths:
    if ep not in sys.path: sys.path.insert(0, ep)
if MEMOTR not in sys.path: sys.path.insert(0, MEMOTR)

test = subprocess.run(
    ['python3', '-c', 'import MultiScaleDeformableAttention; print("ok")'],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': ':'.join(egg_paths + [MEMOTR])}
)
if 'ok' in test.stdout:
    print('  ✅ MultiScaleDeformableAttention importable')
else:
    try:
        import MultiScaleDeformableAttention
        print('  ✅ MultiScaleDeformableAttention import OK')
    except ImportError as e:
        raise RuntimeError(f'Import failed: {e}')

print('\n✅ Cell 3 complete')

Cloning MeMOTR ...
Installing dependencies ...
  ✅ Dependencies installed
Patching CUDA source ...
  ✅ CUDA patches applied
Compiling CUDA ops (~3 min) ...
  ✅ CUDA ops compiled
  ✅ MultiScaleDeformableAttention importable

✅ Cell 3 complete


## Cell 4 — Download MeMOTR DanceTrack Checkpoint

gdown ID: `1MPZJfP91Pb1ThnX5dvxZ7tcjDH8t9hew`

This is the checkpoint trained on DanceTrack — the correct one for zero-shot inference.
The alternative link (`17FxIGgI...`) is for training from scratch and is not needed here.

**Fallback:** if gdown fails, manually download from the Google Drive link, upload the file
to `MyDrive/` and re-run this cell.

In [4]:
import os, subprocess, shutil

MEMOTR   = '/content/MeMOTR'
PTH_PATH = f'{MEMOTR}/memotr_dancetrack.pth'

if os.path.exists(PTH_PATH) and os.path.getsize(PTH_PATH) > 50e6:
    print(f'✅ memotr_dancetrack.pth already present  '
          f'({os.path.getsize(PTH_PATH)/1e6:.1f} MB)')
else:
    DRIVE_CANDIDATES = [
        '/content/drive/MyDrive/memotr_dancetrack.pth',
        '/content/drive/MyDrive/MeMOTR/memotr_dancetrack.pth',
        '/content/drive/MyDrive/weights/memotr_dancetrack.pth',
    ]
    found = next(
        (p for p in DRIVE_CANDIDATES
         if os.path.exists(p) and os.path.getsize(p) > 50e6),
        None
    )
    if found:
        print(f'Copying from Drive: {found} ...')
        shutil.copy(found, PTH_PATH)
        print(f'  ✅ Copied  ({os.path.getsize(PTH_PATH)/1e6:.1f} MB)')
    else:
        print('Downloading via gdown (ID: 1MPZJfP91Pb1ThnX5dvxZ7tcjDH8t9hew) ...')
        r = subprocess.run(
            ['gdown', '1MPZJfP91Pb1ThnX5dvxZ7tcjDH8t9hew', '-O', PTH_PATH],
            capture_output=True, text=True
        )
        ok = os.path.exists(PTH_PATH) and os.path.getsize(PTH_PATH) > 50e6
        if not ok:
            r2 = subprocess.run(
                ['gdown', '--fuzzy',
                 'https://drive.google.com/file/d/1MPZJfP91Pb1ThnX5dvxZ7tcjDH8t9hew/view',
                 '-O', PTH_PATH],
                capture_output=True, text=True
            )
            ok = os.path.exists(PTH_PATH) and os.path.getsize(PTH_PATH) > 50e6
        if not ok:
            if os.path.exists(PTH_PATH): os.remove(PTH_PATH)
            raise RuntimeError(
                'Download failed.\n'
                '1. Open https://drive.google.com/file/d/1MPZJfP91Pb1ThnX5dvxZ7tcjDH8t9hew/view\n'
                '2. Download memotr_dancetrack.pth to your computer\n'
                '3. Upload to Google Drive root (MyDrive/)\n'
                '4. Re-run this cell'
            )
        print(f'  ✅ Downloaded  ({os.path.getsize(PTH_PATH)/1e6:.1f} MB)')

size_mb = os.path.getsize(PTH_PATH) / 1e6
assert size_mb > 50, f'File looks corrupt ({size_mb:.1f} MB)'
print(f'\n✅ memotr_dancetrack.pth ready  ({size_mb:.1f} MB)')

  ✅ Downloaded  (603.5 MB)

✅ memotr_dancetrack.pth ready  (603.5 MB)


## Cell 5 — Clone MOTRv2, Patch CUDA & Compile

MOTRv2 uses the same Deformable Attention CUDA ops as MeMOTR; the same two patches apply.

**Runtime:** ~3 min on T4.

In [5]:
import os, subprocess, sys, glob

MOTRV2 = '/content/MOTRv2'

# ── 1. Clone ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(MOTRV2):
    print('Cloning MOTRv2 ...')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/megvii-research/MOTRv2.git', MOTRV2], check=True)
else:
    print('MOTRv2 already cloned — skipping.')

# ── 2. Install Python dependencies ────────────────────────────────────────────────────────────
print('Installing dependencies ...')
subprocess.run(['pip', 'install', '-q', '-r', f'{MOTRV2}/requirements.txt',
                'cython', 'pycocotools', 'motmetrics', 'gdown', 'ultralytics', 'tqdm'],
               check=True)

# ── 3. Patch PyTorch 2.x API ──────────────────────────────────────────────────────────────────────
cuda_src = f'{MOTRV2}/models/ops/src/cuda/ms_deform_attn_cuda.cu'
print('Patching CUDA source ...')
with open(cuda_src) as f:
    content = f.read()
content = content.replace('value.type()', 'value.scalar_type()')
content = content.replace('value.scalar_type().is_cuda()', 'value.is_cuda()')
with open(cuda_src, 'w') as f:
    f.write(content)
assert 'value.type()' not in content, 'Patch 1 failed'
assert 'scalar_type().is_cuda()' not in content, 'Patch 2 failed'
print('  ✅ CUDA patches applied')

# ── 4. Compile ─────────────────────────────────────────────────────────────────────────────────────
print('Compiling CUDA ops (~3 min) ...')
ops_dir = f'{MOTRV2}/models/ops'
result_c = subprocess.run(
    ['python3', 'setup.py', 'build', 'install'],
    cwd=ops_dir, capture_output=True, text=True
)
if result_c.returncode != 0:
    print(result_c.stdout[-2000:]); print(result_c.stderr[-2000:])
    raise RuntimeError('CUDA compilation failed')
print('  ✅ CUDA ops compiled')

# ── 5. sys.path ──────────────────────────────────────────────────────────────────────────────────────────
egg_paths = (
    glob.glob('/usr/local/lib/python*/dist-packages/MultiScaleDeformableAttention*.egg') +
    glob.glob(f'{ops_dir}/dist/MultiScaleDeformableAttention*.egg')
)
for ep in egg_paths:
    if ep not in sys.path: sys.path.insert(0, ep)
if ops_dir not in sys.path: sys.path.insert(0, ops_dir)
if MOTRV2 not in sys.path: sys.path.insert(0, MOTRV2)

try:
    import MultiScaleDeformableAttention
    print('  ✅ MultiScaleDeformableAttention import OK')
except ImportError:
    test = subprocess.run(
        ['python3', '-c', 'import MultiScaleDeformableAttention; print("ok")'],
        capture_output=True, text=True
    )
    if 'ok' in test.stdout:
        print('  ✅ MultiScaleDeformableAttention compiled (available in Cell 15 subprocess)')
    else:
        raise RuntimeError(f'Import failed. stderr: {test.stderr[:400]}')

print('\n✅ Cell 5 complete')

Cloning MOTRv2 ...
Installing dependencies ...
Patching CUDA source ...
  ✅ CUDA patches applied
Compiling CUDA ops (~3 min) ...
  ✅ CUDA ops compiled
  ✅ MultiScaleDeformableAttention import OK

✅ Cell 5 complete


## Cell 6 — Download MOTRv2 DanceTrack Checkpoint

gdown ID: `1EA4lndu2yQcVgBKR09KfMe5efbf631Th` (~168 MB)

In [6]:
import os, subprocess

MOTRV2   = '/content/MOTRv2'
PTH_PATH = f'{MOTRV2}/pretrained/motrv2.pth'
os.makedirs(f'{MOTRV2}/pretrained', exist_ok=True)

if not os.path.exists(PTH_PATH):
    print('Downloading motrv2.pth (~168 MB) ...')
    result = subprocess.run(
        ['gdown', '1EA4lndu2yQcVgBKR09KfMe5efbf631Th', '-O', PTH_PATH],
        capture_output=True, text=True
    )
    if result.returncode != 0 or not os.path.exists(PTH_PATH):
        print(result.stdout); print(result.stderr)
        raise RuntimeError('Download failed — check gdown output above')
else:
    print('motrv2.pth already present — skipping.')

print(f'✅ motrv2.pth ready  ({os.path.getsize(PTH_PATH)/1e6:.1f} MB)')

✅ motrv2.pth ready  (168.1 MB)


## Cell 7 — Clone & Patch TrackEval

In [7]:
import os, glob, re, subprocess

TRACKEVAL = '/content/TrackEval'

if not os.path.exists(TRACKEVAL):
    print('Cloning TrackEval ...')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/JonathonLuiten/TrackEval.git',
                    TRACKEVAL], check=True)
else:
    print('TrackEval already present — skipping.')

patched = 0
for fp in glob.glob(f'{TRACKEVAL}/**/*.py', recursive=True):
    with open(fp, 'r', encoding='utf-8') as f: d = f.read()
    d2 = re.sub(r'np\.(float|int|bool)(?!\d|_)', r'\1', d)
    if d != d2:
        with open(fp, 'w', encoding='utf-8') as f: f.write(d2)
        patched += 1
print(f'Patched {patched} file(s) for NumPy 2.x compatibility')
print('✅ TrackEval ready')

Cloning TrackEval ...
Patched 19 file(s) for NumPy 2.x compatibility
✅ TrackEval ready


## Cell 8 — Configuration

**Edit only this cell** if you need to change any parameter.

In [8]:
# =============================================================================
#  ALL PARAMETERS — edit only here
# =============================================================================

MEMOTR    = '/content/MeMOTR'
MOTRV2    = '/content/MOTRv2'
TRACKEVAL = '/content/TrackEval'

# Image resolution — original camera sensor resolution
IMG_W, IMG_H = 2064, 1544

# Sequence names and exact frame counts
SEQ_LIMITS = {
    'sun_glare_0': 836,
    'sun_glare_1': 247,
    'sun_glare_2': 323,
    'sun_glare_3': 1046,
}

# TartuGlare class ID → COCO class ID mapping
# TartuGlare: 0=Pedestrian  1=Cyclist  2=Car  3=Motorcycle  4=Bus   5=Truck
# COCO:       0=person      1=bicycle  2=car  3=motorcycle   5=bus   7=truck
COCO_CLASSES = [0, 1, 2, 3, 5, 7]

# YOLO model for MOTRv2 proposal generation (same as TbD notebook — fair comparison)
MODEL_NAME = 'yolo11x.pt'
CONF_YOLO  = 0.20    # low — MOTRv2 transformer re-scores proposals internally
IMGSZ      = 1280

# ── Evaluation filters — identical across all trackers ───────────────────────────────────
CONF_EVAL  = 0.50   # MOT17 / MOT20 / DanceTrack evaluation standard
MIN_HEIGHT = 50     # px — DanceTrack (CVPR 2022) standard

MEMOTR_OUTPUT = '/content/memotr_results'
MOTRV2_OUTPUT = '/content/motrv2_results'
import os
os.makedirs(MEMOTR_OUTPUT, exist_ok=True)
os.makedirs(MOTRV2_OUTPUT, exist_ok=True)

# =============================================================================

print('Configuration loaded ✅')
print(f'  Image size      : {IMG_W} × {IMG_H}')
print(f'  COCO classes    : {COCO_CLASSES}')
print(f'  YOLO model      : {MODEL_NAME}  conf={CONF_YOLO}  imgsz={IMGSZ}')
print(f'  Eval filters    : conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px  (DanceTrack standard)')
print(f'  Gradient ckpt   : {USE_CHECKPOINT}  (required for T4 15.6 GB VRAM)')

Configuration loaded ✅
  Image size      : 2064 × 1544
  COCO classes    : [0, 1, 2, 3, 5, 7]
  YOLO model      : yolo11x.pt  conf=0.2  imgsz=1280
  Eval filters    : conf≥0.5  min_height≥50px  (DanceTrack standard)
  Gradient ckpt   : True  (required for T4 15.6 GB VRAM)


## Cell 9 — Detect Label Format & Build MOTChallenge Ground Truth

Shared GT is built once and used by both MeMOTR and MOTRv2.

In [9]:
import os, glob as _glob

labels_are_normalized = None
for seq, lbl_dir in LBL_DIRS.items():
    if not lbl_dir: continue
    txts = sorted(_glob.glob(f'{lbl_dir}/*.txt'))
    vals = []
    for tf in txts[:5]:
        with open(tf) as f:
            for line in f:
                p = line.strip().split()
                if len(p) >= 6:
                    try: vals.extend(float(x) for x in p[2:6])
                    except ValueError: pass
        if len(vals) >= 40: break
    if vals:
        mx = max(vals)
        labels_are_normalized = mx <= 1.5
        fmt = 'NORMALIZED [0–1]' if labels_are_normalized else f'ABSOLUTE PIXELS (max={mx:.0f})'
        print(f'Label format: max={mx:.4f}  →  {fmt}')
        break
if labels_are_normalized is None:
    labels_are_normalized = True
    print('Could not sample labels — defaulting to NORMALIZED')

BASE_GT = f'{TRACKEVAL}/data/gt/mot_challenge/TartuGlare-train'

for seq, limit in SEQ_LIMITS.items():
    lbl_dir = LBL_DIRS.get(seq)
    if not lbl_dir: continue
    gt_folder = f'{BASE_GT}/{seq}/gt'
    os.makedirs(gt_folder, exist_ok=True)
    with open(f'{BASE_GT}/{seq}/seqinfo.ini', 'w') as f:
        f.write(f'[Sequence]\nname={seq}\nseqLength={limit}\n'
                f'imWidth={IMG_W}\nimHeight={IMG_H}\nimExt=.jpg\n')
    txt_files = sorted(_glob.glob(f'{lbl_dir}/*.txt'))
    rows = 0
    with open(f'{gt_folder}/gt.txt', 'w') as out_f:
        for idx, txt in enumerate(txt_files):
            f_id = idx + 1
            if f_id > limit: break
            with open(txt) as in_f:
                for line in in_f:
                    p = line.strip().split()
                    if len(p) < 5: continue
                    cls = int(p[0])
                    if cls not in range(6): continue
                    if len(p) >= 6:
                        o_id = int(p[1]); raw = list(map(float, p[2:6]))
                    else:
                        o_id = (f_id*10000+int(float(p[1])*10000))%99999+1
                        raw = list(map(float, p[1:5]))
                    cx, cy, nw, nh = raw
                    if labels_are_normalized:
                        aw=nw*IMG_W; ah=nh*IMG_H
                        al=cx*IMG_W-aw/2; at=cy*IMG_H-ah/2
                    else:
                        aw,ah=nw,nh
                        al=cx-aw/2 if cx>aw/2 else cx
                        at=cy-ah/2 if cy>ah/2 else cy
                    al=max(0.,al); at=max(0.,at)
                    aw=min(aw,IMG_W-al); ah=min(ah,IMG_H-at)
                    if aw<=0 or ah<=0: continue
                    out_f.write(f'{f_id},{o_id},{al:.2f},{at:.2f},{aw:.2f},{ah:.2f},1,1,1\n')
                    rows += 1
    print(f'  {seq}: {rows} GT rows  ({len(txt_files)} label files)')
print('\n✅ Ground truth built')

Label format: max=0.9176  →  NORMALIZED [0–1]
  sun_glare_0: 3668 GT rows  (836 label files)
  sun_glare_1: 1699 GT rows  (247 label files)
  sun_glare_2: 1035 GT rows  (323 label files)
  sun_glare_3: 4194 GT rows  (1046 label files)

✅ Ground truth built


## Cell 10 — Run MeMOTR Inference

MeMOTR is invoked via `main.py` in `--mode submit`. It reads images from a DanceTrack-style
directory structure built via symlinks (no data duplication).

Key flags:
- `--use-checkpoint`: gradient checkpointing — mandatory for T4 (15.6 GB VRAM)
- `--resume`: path to the DanceTrack-pretrained checkpoint

**Runtime:** ~30–45 min on T4.

In [10]:
import os, subprocess, sys, glob, yaml, shutil

MEMOTR        = '/content/MeMOTR'
MEMOTR_OUTPUT = '/content/memotr_results'
PTH_MEMOTR    = f'{MEMOTR}/memotr_dancetrack.pth'

if MEMOTR not in sys.path: sys.path.insert(0, MEMOTR)
for ep in glob.glob('/usr/local/lib/python*/dist-packages/MultiScaleDeformableAttention*.egg'):
    if ep not in sys.path: sys.path.insert(0, ep)

# ── Build DanceTrack image structure (symlinks) ───────────────────────────────────────
DANCE_TEST = f'{MEMOTR}/data/DanceTrack/test'
os.makedirs(DANCE_TEST, exist_ok=True)

seqmap_path = f'{MEMOTR}/data/DanceTrack/test_seqmap.txt'
with open(seqmap_path, 'w') as f:
    f.write('name\n')
    for seq in SEQ_LIMITS:
        f.write(f'{seq}\n')

for seq, limit in SEQ_LIMITS.items():
    img_dir = IMG_DIRS.get(seq)
    if not img_dir: print(f'  ❌ {seq}: image dir not found'); continue
    img1_dir = f'{DANCE_TEST}/{seq}/img1'
    os.makedirs(img1_dir, exist_ok=True)
    with open(f'{DANCE_TEST}/{seq}/seqinfo.ini', 'w') as f:
        f.write(f'[Sequence]\nname={seq}\nseqLength={limit}\n'
                f'imWidth={IMG_W}\nimHeight={IMG_H}\nimExt=.jpg\nimDir=img1\n')
    img_files = sorted(
        glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    )[:limit]
    for idx, src in enumerate(img_files):
        dst = f'{img1_dir}/{idx+1:06d}.jpg'
        if not os.path.exists(dst):
            if src.endswith('.jpg'):
                os.symlink(os.path.abspath(src), dst)
            else:
                shutil.copy(src, dst)
    print(f'  ✅ {seq}: {len(img_files)} frames staged')

# ── Patch config ───────────────────────────────────────────────────────────────────────────────────────────
config_path = f'{MEMOTR}/configs/train_dancetrack.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)
cfg['SUBMIT_DIR']        = MEMOTR_OUTPUT
cfg['DATA_ROOT']         = f'{MEMOTR}/data'
cfg['SUBMIT_DATA_SPLIT'] = 'test'
cfg['SUBMIT_MODEL']      = PTH_MEMOTR
cfg['USE_CHECKPOINT']    = USE_CHECKPOINT
cfg['BATCH_SIZE']        = 1
patched_config = f'{MEMOTR}/configs/submit_tartuglare.yaml'
with open(patched_config, 'w') as f:
    yaml.dump(cfg, f)
train_dir = f'{MEMOTR_OUTPUT}/train'
os.makedirs(train_dir, exist_ok=True)
with open(f'{train_dir}/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

# ── Run inference ─────────────────────────────────────────────────────────────────────────────────────────────
cmd = [
    sys.executable, f'{MEMOTR}/main.py',
    '--mode',              'submit',
    '--config-path',       patched_config,
    '--resume',            PTH_MEMOTR,
    '--data-root',         f'{MEMOTR}/data',
    '--outputs-dir',       MEMOTR_OUTPUT,
    '--submit-dir',        MEMOTR_OUTPUT,
    '--submit-model',      PTH_MEMOTR,
    '--submit-data-split', 'test',
]
if USE_CHECKPOINT:
    cmd.append('--use-checkpoint')

log_path = f'{MEMOTR_OUTPUT}/run.log'
print(f'Running MeMOTR — output → {log_path}')
with open(log_path, 'w') as log_f:
    result_memotr = subprocess.run(
        cmd, cwd=MEMOTR, stdout=log_f, stderr=subprocess.STDOUT
    )

print('\n── run.log (last 40 lines) ──')
with open(log_path) as f:
    lines = f.readlines()
for line in lines[-40:]:
    print(line, end='')

# Read log content so save cell can write it to Drive
with open(log_path) as f:
    memotr_log_text = f.read()

if result_memotr.returncode != 0:
    raise RuntimeError(f'Exit code {result_memotr.returncode} — see run.log above.')
print('\n✅ MeMOTR inference complete')

  ✅ sun_glare_0: 836 frames staged
  ✅ sun_glare_1: 247 frames staged
  ✅ sun_glare_2: 323 frames staged
  ✅ sun_glare_3: 1046 frames staged
Running MeMOTR — output → /content/memotr_results/run.log

── run.log (last 40 lines) ──
Submit seq: sun_glare_1: 100%|██████████| 247/247 [00:59<00:00,  4.15it/s]

✅ MeMOTR inference complete


## Cell 11 — Convert MeMOTR Output & Apply Evaluation Filters

Applies the same three filters as the TbD notebook (`conf ≥ 0.50`, `min_height ≥ 50 px`, clamp)
and writes results to the TrackEval tracker input directory.

In [11]:
import os, glob, collections

MEMOTR_OUTPUT = '/content/memotr_results'
TRACKER_NAME  = 'memotr'

OUT_DIR_MEMOTR = (f'{TRACKEVAL}/data/trackers/mot_challenge/'
                  f'TartuGlare-train/{TRACKER_NAME}/data')
os.makedirs(OUT_DIR_MEMOTR, exist_ok=True)

GT_DETS = {'sun_glare_0':3668,'sun_glare_1':1699,'sun_glare_2':1035,'sun_glare_3':4194}

print(f'Filters : conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px  clamp {IMG_W}×{IMG_H}\n')

for seq in SEQ_LIMITS:
    candidates = (
        glob.glob(f'{MEMOTR_OUTPUT}/{seq}.txt') +
        glob.glob(f'{MEMOTR_OUTPUT}/**/{seq}.txt', recursive=True)
    )
    if not candidates:
        print(f'  ❌ {seq}: output not found in {MEMOTR_OUTPUT}')
        continue
    in_path = candidates[0]
    kept    = []
    dropped = collections.Counter()

    with open(in_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            p = line.split(',')
            if len(p) < 6: continue
            try:
                frame  = int(float(p[0]))
                obj_id = int(float(p[1]))
                l, t, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
                conf = (float(p[6]) if len(p) > 6
                        and p[6].strip() not in ('', '-1', '-1.0')
                        else 1.0)
            except (ValueError, IndexError):
                continue

            if conf < CONF_EVAL:
                dropped['conf'] += 1; continue
            l2=max(0.,l); t2=max(0.,t)
            w2=min(w,IMG_W-l2); h2=min(h,IMG_H-t2)
            if h2 < MIN_HEIGHT:
                dropped['min_h'] += 1; continue
            if w2 < 5 or h2 < 5:
                dropped['degen'] += 1; continue

            kept.append(
                f'{frame},{obj_id},{l2:.2f},{t2:.2f},'
                f'{w2:.2f},{h2:.2f},{conf:.4f},-1,-1,-1\n'
            )

    out_path = f'{OUT_DIR_MEMOTR}/{seq}.txt'
    with open(out_path, 'w') as f:
        f.writelines(kept)

    ratio  = len(kept) / GT_DETS[seq]
    status = '✅' if 0.03 <= ratio <= 1.40 else '⚠ '
    print(f'  {status} {seq}: {len(kept)} kept  GT={GT_DETS[seq]}  ratio={ratio:.3f}  '
          f'| dropped: conf={dropped["conf"]}  min_h={dropped["min_h"]}  degen={dropped["degen"]}')

print('\n✅ MeMOTR output filtered')

Filters : conf≥0.5  min_height≥50px  clamp 2064×1544

  ✅ sun_glare_0: 301 kept  GT=3668  ratio=0.082  | dropped: conf=0  min_h=47  degen=0
  ✅ sun_glare_1: 97 kept  GT=1699  ratio=0.057  | dropped: conf=0  min_h=2  degen=0
  ✅ sun_glare_2: 41 kept  GT=1035  ratio=0.040  | dropped: conf=0  min_h=17  degen=0
  ✅ sun_glare_3: 312 kept  GT=4194  ratio=0.074  | dropped: conf=0  min_h=42  degen=0

✅ MeMOTR output filtered


## Cell 12 — Run TrackEval (MeMOTR)

In [12]:
TRACKER_NAME = 'memotr'

import os, subprocess, sys

seqmap_dir = f'{TRACKEVAL}/data/gt/mot_challenge/seqmaps'
os.makedirs(seqmap_dir, exist_ok=True)
with open(f'{seqmap_dir}/TartuGlare-train.txt', 'w') as f:
    f.write('name\nsun_glare_0\nsun_glare_1\nsun_glare_2\nsun_glare_3\n')

cmd = [
    sys.executable,
    f'{TRACKEVAL}/scripts/run_mot_challenge.py',
    '--BENCHMARK',          'TartuGlare',
    '--SPLIT_TO_EVAL',      'train',
    '--TRACKERS_TO_EVAL',   TRACKER_NAME,
    '--METRICS',            'HOTA', 'CLEAR', 'Identity',
    '--USE_PARALLEL',       'False',
    '--NUM_PARALLEL_CORES', '1',
]

print('Running TrackEval ...\n')
result = subprocess.run(cmd, cwd=TRACKEVAL, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    print('⚠  Non-zero exit — check stderr above')

memotr_result = result

Running TrackEval ...


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 1                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mo

## Cell 13 — Pre-download YOLO Model

Downloads YOLO11x weights before the proposal-generation loop.

In [13]:
from ultralytics import YOLO
print(f'Downloading / verifying {MODEL_NAME} ...')
_tmp = YOLO(MODEL_NAME)
del _tmp
print(f'✅ {MODEL_NAME} ready')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ yolo11x.pt ready


## Cell 14 — Generate YOLO Proposals & Build DanceTrack Structure

MOTRv2 requires detector proposals in a `det_db.json` file (DanceTrack format).
This cell runs YOLO11x at `conf=0.20` — the low threshold ensures recall is maximised
for the transformer's internal re-scoring.

**Runtime:** ~10–15 min on T4.

In [14]:
import os, glob, json, shutil, collections
from tqdm.notebook import tqdm
from ultralytics import YOLO
import torch

MOTRV2 = '/content/MOTRv2'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device : {"GPU — " + torch.cuda.get_device_name(0) if DEVICE==0 else "CPU"}')
print(f'Conf   : {CONF_YOLO}  (low — MOTRv2 re-scores proposals internally)')
print()

DANCE_ROOT = f'{MOTRV2}/datasets/DanceTrack/test'
os.makedirs(DANCE_ROOT, exist_ok=True)

det_db = {}

for seq, limit in SEQ_LIMITS.items():
    img_dir = IMG_DIRS.get(seq)
    if not img_dir:
        print(f'❌ {seq}: image dir not found — skipping'); continue

    img_files = sorted(glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png'))
    total = min(len(img_files), limit)
    print(f'── {seq}  ({total} frames) ──')

    img1_dir = f'{DANCE_ROOT}/{seq}/img1'
    os.makedirs(img1_dir, exist_ok=True)
    with open(f'{DANCE_ROOT}/{seq}/seqinfo.ini', 'w') as f:
        f.write(f'[Sequence]\nname={seq}\nseqLength={total}\n'
                f'imWidth={IMG_W}\nimHeight={IMG_H}\nimExt=.jpg\nimDir=img1\n')
    for idx, src in enumerate(img_files[:total]):
        dst = f'{img1_dir}/{idx+1:06d}.jpg'
        if not os.path.exists(dst):
            if src.endswith('.jpg'): os.symlink(os.path.abspath(src), dst)
            else: shutil.copy(src, dst)

    model = YOLO(MODEL_NAME)
    results = model.predict(
        source=img_dir, classes=COCO_CLASSES, conf=CONF_YOLO,
        imgsz=IMGSZ, device=DEVICE, verbose=False, stream=True,
    )

    n_dets = 0
    pbar = tqdm(total=total, unit='frame', desc=seq, leave=True)

    for frame_idx, r in enumerate(results):
        if frame_idx >= limit: break
        pbar.update(1)
        key = f'DanceTrack/test/{seq}/img1/{frame_idx+1:06d}.txt'  # submit_dance.py does f_path[:-4]+'.txt'
        frame_dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            boxes = r.boxes.xywh.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()
            for box, conf in zip(boxes, confs):
                x_c, y_c, w, h = box
                x_min=max(0., x_c-w/2); y_min=max(0., y_c-h/2)
                w2=min(w, IMG_W-x_min); h2=min(h, IMG_H-y_min)
                if w2 < 5 or h2 < 5: continue
                frame_dets.append(f'{x_min:.1f},{y_min:.1f},{w2:.1f},{h2:.1f},{conf:.4f}')
                n_dets += 1
        det_db[key] = frame_dets

    pbar.close()
    print(f'  {n_dets} detections across {total} frames\n')

det_db_path = f'{MOTRV2}/det_db_motrv2.json'
with open(det_db_path, 'w') as f:
    json.dump(det_db, f)
print(f'✅ det_db_motrv2.json saved  ({len(det_db)} frame entries)')

Device : GPU — Tesla T4
Conf   : 0.2  (low — MOTRv2 re-scores proposals internally)

── sun_glare_0  (836 frames) ──


sun_glare_0:   0%|          | 0/836 [00:00<?, ?frame/s]

  8304 detections across 836 frames

── sun_glare_1  (247 frames) ──


sun_glare_1:   0%|          | 0/247 [00:00<?, ?frame/s]

  4066 detections across 247 frames

── sun_glare_2  (323 frames) ──


sun_glare_2:   0%|          | 0/323 [00:00<?, ?frame/s]

  2605 detections across 323 frames

── sun_glare_3  (1046 frames) ──


sun_glare_3:   0%|          | 0/1046 [00:00<?, ?frame/s]

  9211 detections across 1046 frames

✅ det_db_motrv2.json saved  (2452 frame entries)


## Cell 15 — Run MOTRv2 Inference

MOTRv2 processes all four sequences in a single call to `submit_dance.py`.

**Runtime:** ~15–25 min on T4.

In [15]:
import os, subprocess, sys, glob

MOTRV2        = '/content/MOTRv2'
MOTRV2_OUTPUT = '/content/motrv2_results'
os.makedirs(MOTRV2_OUTPUT, exist_ok=True)

MOT_PATH = f'{MOTRV2}/datasets/MOT17'
os.makedirs(MOT_PATH, exist_ok=True)

# ── Fix 1: Clean up stale/broken symlinks before re-creating them ──────────────
dance_link = f'{MOT_PATH}/DanceTrack'
if os.path.islink(dance_link) and not os.path.exists(dance_link):
    os.unlink(dance_link)   # remove broken symlink
if not os.path.exists(dance_link):
    os.symlink(f'{MOTRV2}/datasets/DanceTrack', dance_link)

det_db_link = f'{MOT_PATH}/det_db_motrv2.json'
if os.path.islink(det_db_link) and not os.path.exists(det_db_link):
    os.unlink(det_db_link)  # remove broken symlink
if not os.path.exists(det_db_link):
    os.symlink(f'{MOTRV2}/det_db_motrv2.json', det_db_link)

# ── Fix 2: Pass MultiScaleDeformableAttention egg paths to subprocess via PYTHONPATH ──
ops_dir   = f'{MOTRV2}/models/ops'
egg_paths = (
    glob.glob('/usr/local/lib/python*/dist-packages/MultiScaleDeformableAttention*.egg') +
    glob.glob(f'{ops_dir}/dist/MultiScaleDeformableAttention*.egg')
)
extra_paths = egg_paths + [MOTRV2, ops_dir]
env = {
    **os.environ,
    'PYTHONPATH': ':'.join(extra_paths + [os.environ.get('PYTHONPATH', '')])
}

cmd = [
    sys.executable,
    f'{MOTRV2}/submit_dance.py',
    '--meta_arch',              'motr',
    '--dataset_file',           'e2e_dance',
    '--with_box_refine',
    '--query_interaction_layer','QIMv2',
    '--num_queries',            '10',
    '--use_checkpoint',
    '--resume',                 f'{MOTRV2}/pretrained/motrv2.pth',
    '--mot_path',               MOT_PATH,
    '--output_dir',             MOTRV2_OUTPUT,
    '--batch_size',             '1',
    '--det_db',                 'det_db_motrv2.json',
    '--sampler_lengths',        '2', '3', '4', '5',
]

# ── Fix 3: Capture output to log file so errors are always visible ──────────────
log_path = f'{MOTRV2_OUTPUT}/run.log'
print(f'Running MOTRv2 inference — output → {log_path}')
with open(log_path, 'w') as log_f:
    result_motrv2 = subprocess.run(
        cmd, cwd=MOTRV2,
        stdout=log_f, stderr=subprocess.STDOUT,
        text=True, env=env
    )

# ── Always show last 60 lines of log so failures are immediately visible ─────────
print('\n── run.log (last 60 lines) ──')
with open(log_path) as f:
    lines = f.readlines()
for line in lines[-60:]:
    print(line, end='')

# Read log text so Cell 19 can save it
with open(log_path) as f:
    motrv2_log_text = f.read()

if result_motrv2.returncode != 0:
    raise RuntimeError(
        f'MOTRv2 failed with exit code {result_motrv2.returncode}\n'
        f'Last 20 log lines:\n' + ''.join(lines[-20:])
    )

print('\n✅ MOTRv2 inference complete. Output files:')
for fname in sorted(os.listdir(MOTRV2_OUTPUT)):
    size = os.path.getsize(f'{MOTRV2_OUTPUT}/{fname}')
    print(f'  {fname}  ({size:,} bytes)')


Running MOTRv2 inference — output → /content/motrv2_results/run.log

── run.log (last 60 lines) ──
100%|██████████| 247/247 [00:49<00:00,  5.03it/s]
totally 197 dts 0 occlusion dts

✅ MOTRv2 inference complete. Output files:
  run.log  (152,177 bytes)
  submit  (4,096 bytes)


## Cell 16 — Convert MOTRv2 Output & Apply Evaluation Filters

Reads MOTRv2's `.txt` output, applies the same three filters used for all other trackers,
and writes to the TrackEval tracker input directory.

In [16]:
import os, glob, collections

MOTRV2_OUTPUT = '/content/motrv2_results'
TRACKER_NAME  = 'motrv2'

OUT_DIR_MOTRV2 = (f'{TRACKEVAL}/data/trackers/mot_challenge/'
                  f'TartuGlare-train/{TRACKER_NAME}/data')
os.makedirs(OUT_DIR_MOTRV2, exist_ok=True)

GT_DETS = {'sun_glare_0':3668,'sun_glare_1':1699,'sun_glare_2':1035,'sun_glare_3':4194}

print(f'Filters : conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px  clamp {IMG_W}×{IMG_H}\n')

for seq in SEQ_LIMITS:
    in_path = f'{MOTRV2_OUTPUT}/{seq}.txt'
    if not os.path.exists(in_path):
        alts = glob.glob(f'{MOTRV2_OUTPUT}/**/{seq}.txt', recursive=True)
        if alts:
            in_path = alts[0]
        else:
            print(f'  ❌ {seq}: output not found  (files: {os.listdir(MOTRV2_OUTPUT)})')
            continue

    kept    = []
    dropped = collections.Counter()

    with open(in_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            p = line.split(',')
            if len(p) < 6: continue
            frame  = int(float(p[0]))
            obj_id = int(float(p[1]))
            l, t, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
            conf = float(p[6]) if len(p) > 6 and p[6].strip() not in ('-1','-1.0','') else 1.0

            if conf < CONF_EVAL:
                dropped['conf'] += 1; continue
            l2=max(0.,l); t2=max(0.,t)
            w2=min(w,IMG_W-l2); h2=min(h,IMG_H-t2)
            if h2 < MIN_HEIGHT:
                dropped['min_h'] += 1; continue
            if w2 < 5 or h2 < 5:
                dropped['degen'] += 1; continue

            kept.append(
                f'{frame},{obj_id},{l2:.2f},{t2:.2f},'
                f'{w2:.2f},{h2:.2f},{conf:.4f},-1,-1,-1\n'
            )

    out_path = f'{OUT_DIR_MOTRV2}/{seq}.txt'
    with open(out_path, 'w') as f:
        f.writelines(kept)

    ratio  = len(kept) / GT_DETS[seq]
    status = '✅' if 0.70 <= ratio <= 1.30 else '⚠ '
    print(f'  {status} {seq}: {len(kept)} dets kept  GT={GT_DETS[seq]}  ratio={ratio:.3f}  '
          f'| dropped: conf={dropped["conf"]}  min_h={dropped["min_h"]}  degen={dropped["degen"]}')

print('\n✅ MOTRv2 output filtered')

Filters : conf≥0.5  min_height≥50px  clamp 2064×1544

  ⚠  sun_glare_0: 405 dets kept  GT=3668  ratio=0.110  | dropped: conf=0  min_h=78  degen=0
  ⚠  sun_glare_1: 148 dets kept  GT=1699  ratio=0.087  | dropped: conf=0  min_h=49  degen=0
  ⚠  sun_glare_2: 259 dets kept  GT=1035  ratio=0.250  | dropped: conf=0  min_h=15  degen=0
  ⚠  sun_glare_3: 548 dets kept  GT=4194  ratio=0.131  | dropped: conf=0  min_h=92  degen=0

✅ MOTRv2 output filtered


## Cell 17 — Coordinate Sanity Check

Verifies that GT and MOTRv2 boxes share the same pixel coordinate space.
Note: IDs will not match (E2E trackers assign their own IDs), so the IoU check
uses a best-match search across all frame-1 box pairs rather than an exact key lookup.

In [17]:
print(f'Sanity check — sun_glare_0, first 5 lines each')
print(f'Expected: l ∈ [0, {IMG_W}]   t ∈ [0, {IMG_H}]\n')

BASE_GT   = f'{TRACKEVAL}/data/gt/mot_challenge/TartuGlare-train'
gt_path   = f'{BASE_GT}/sun_glare_0/gt/gt.txt'
tr_path   = f'{OUT_DIR_MOTRV2}/sun_glare_0.txt'

for label, path in [('GT     ', gt_path), ('MOTRv2 ', tr_path)]:
    print(f'  [{label}]')
    try:
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= 5: break
                p = line.strip().split(',')
                lv,tv,wv,hv = float(p[2]),float(p[3]),float(p[4]),float(p[5])
                ok = '✅' if 0<=lv<=IMG_W and 0<=tv<=IMG_H else '❌'
                print(f'    {ok} frame={p[0]:>4}  id={p[1]:>5}  '
                      f'l={lv:7.1f}  t={tv:7.1f}  w={wv:7.1f}  h={hv:7.1f}')
    except FileNotFoundError:
        print(f'    ❌ Not found: {path}')
    print()

gt_f1, tr_f1 = {}, {}
with open(gt_path) as f:
    for line in f:
        p=line.strip().split(',')
        if int(p[0])==1: gt_f1[int(p[1])]=tuple(float(x) for x in p[2:6])
with open(tr_path) as f:
    for line in f:
        p=line.strip().split(',')
        if int(p[0])==1: tr_f1[int(p[1])]=tuple(float(x) for x in p[2:6])

best_iou = 0.0
for gbox in gt_f1.values():
    for tbox in tr_f1.values():
        ix=max(gbox[0],tbox[0]); iy=max(gbox[1],tbox[1])
        ix2=min(gbox[0]+gbox[2],tbox[0]+tbox[2])
        iy2=min(gbox[1]+gbox[3],tbox[1]+tbox[3])
        inter=max(0,ix2-ix)*max(0,iy2-iy)
        union=gbox[2]*gbox[3]+tbox[2]*tbox[3]-inter
        if union>0: best_iou=max(best_iou,inter/union)
print(f'Best IoU frame 1: {best_iou:.3f}')
if best_iou >= 0.20: print('  ✅ Coordinates broadly aligned (IDs differ — expected for E2E)')
else: print('  ❌ Very low IoU — check coordinate space')

Sanity check — sun_glare_0, first 5 lines each
Expected: l ∈ [0, 2064]   t ∈ [0, 1544]

  [GT     ]
    ✅ frame=   1  id=    1  l= 1156.5  t= 1296.4  w=  376.9  h=  240.6
    ✅ frame=   1  id=    2  l= 1130.1  t= 1279.0  w=   68.1  h=   54.5
    ✅ frame=   1  id=    3  l=  415.9  t= 1237.5  w=   52.0  h=  166.6
    ✅ frame=   1  id=    4  l= 1192.5  t= 1273.7  w=   52.8  h=   40.1
    ✅ frame=   1  id=    5  l=  513.0  t= 1256.1  w=   43.9  h=  142.7

  [MOTRv2 ]
    ✅ frame=   1  id=  195  l= 1152.3  t= 1299.5  w=  384.6  h=  239.4
    ✅ frame=   1  id=  196  l= 1129.5  t= 1281.6  w=   70.3  h=   56.0
    ✅ frame=   1  id=  197  l=  413.9  t= 1241.2  w=   54.4  h=  165.7
    ✅ frame=   1  id=  198  l=  466.4  t= 1245.8  w=   55.0  h=  162.7
    ✅ frame=   1  id=  199  l=  515.3  t= 1260.2  w=   44.3  h=  141.7

Best IoU frame 1: 0.961
  ✅ Coordinates broadly aligned (IDs differ — expected for E2E)


## Cell 18 — Run TrackEval (MOTRv2)

In [18]:
TRACKER_NAME = 'motrv2'

import os, subprocess, sys

seqmap_dir = f'{TRACKEVAL}/data/gt/mot_challenge/seqmaps'
os.makedirs(seqmap_dir, exist_ok=True)
with open(f'{seqmap_dir}/TartuGlare-train.txt', 'w') as f:
    f.write('name\nsun_glare_0\nsun_glare_1\nsun_glare_2\nsun_glare_3\n')

cmd = [
    sys.executable,
    f'{TRACKEVAL}/scripts/run_mot_challenge.py',
    '--BENCHMARK',          'TartuGlare',
    '--SPLIT_TO_EVAL',      'train',
    '--TRACKERS_TO_EVAL',   TRACKER_NAME,
    '--METRICS',            'HOTA', 'CLEAR', 'Identity',
    '--USE_PARALLEL',       'False',
    '--NUM_PARALLEL_CORES', '1',
]

print('Running TrackEval ...\n')
result = subprocess.run(cmd, cwd=TRACKEVAL, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    print('⚠  Non-zero exit — check stderr above')

motrv2_result = result

Running TrackEval ...


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 1                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mo

## Cell 19 — Save All Results to Drive

Saves metric text files and raw tracker output for both MeMOTR and MOTRv2.

In [19]:
import shutil, os

for tracker_name, res, out_dir in [
    ('memotr',  memotr_result,  OUT_DIR_MEMOTR),
    ('motrv2',  motrv2_result,  OUT_DIR_MOTRV2),
]:
    out_txt = f'/content/drive/MyDrive/{tracker_name}_final_metrics.txt'
    with open(out_txt, 'w') as f:
        f.write('=' * 60 + '\n')
        f.write(f'  {tracker_name.upper()} — SolarDrive / TartuGlare Dataset\n')
        if tracker_name == 'memotr':
            f.write(f'  Checkpoint : memotr_dancetrack.pth (ICCV 2023)\n')
            f.write(f'  Arch       : End-to-end, long-term memory transformer\n')
            f.write(f'  Grad ckpt  : {USE_CHECKPOINT}  (T4 memory optimisation)\n')
        else:
            f.write(f'  Model (proposals) : {MODEL_NAME}  conf={CONF_YOLO}  imgsz={IMGSZ}\n')
            f.write(f'  Tracker           : MOTRv2  (pretrained/motrv2.pth)\n')
            f.write(f'  COCO cls          : {COCO_CLASSES}\n')
        f.write(f'  Eval filter: conf≥{CONF_EVAL}  min_height≥{MIN_HEIGHT}px\n')
        f.write(f'  Resolution : {IMG_W}×{IMG_H}\n')
        f.write('=' * 60 + '\n\n')
        if tracker_name == 'memotr':
            f.write(memotr_log_text)
        else:
            f.write(res.stdout if res.stdout else '')
            if res.stderr:
                f.write('\n\nSTDERR / WARNINGS:\n')
                f.write(res.stderr)
    print(f'✅ Metrics saved: {out_txt}')

    tracker_drive = f'/content/drive/MyDrive/{tracker_name}_tracker_output'
    if os.path.exists(tracker_drive):
        shutil.rmtree(tracker_drive)
    shutil.copytree(out_dir, tracker_drive)
    print(f'✅ Tracker files saved: {tracker_drive}')
    print()

# Also save the MOTRv2 det_db for full reproducibility
shutil.copy(
    det_db_path,
    '/content/drive/MyDrive/motrv2_det_db.json'
)
print('✅ det_db_motrv2.json saved to MyDrive/')

✅ Metrics saved: /content/drive/MyDrive/memotr_final_metrics.txt
✅ Tracker files saved: /content/drive/MyDrive/memotr_tracker_output

✅ Metrics saved: /content/drive/MyDrive/motrv2_final_metrics.txt
✅ Tracker files saved: /content/drive/MyDrive/motrv2_tracker_output

✅ det_db_motrv2.json saved to MyDrive/
